# 💡 HyDE — Hypothetical Document Embeddings

## The Big Idea

**Normal search:** embed the *question*, find documents similar to the question.  
**HyDE:** generate a *hypothetical answer*, embed that, find documents similar to the answer.

Why does this work? Because **documents look like answers, not questions.**

```
❌ NORMAL SEARCH:

   Query:    "What causes overfitting in neural networks?"
                              ↓ embed
   Vector:   [question-like vector]
                              ↓ search
   Document: "Overfitting occurs when a model memorizes training data..."
              [answer-like vector]   ← MISMATCH in vector space!


✅ HyDE:

   Query:    "What causes overfitting in neural networks?"
                              ↓
              LLM generates hypothetical answer:
              "Overfitting is caused by too many parameters, insufficient
               training data, and lack of regularization..."
                              ↓ embed
   Vector:   [answer-like vector]
                              ↓ search
   Document: "Overfitting occurs when a model memorizes training data..."
              [answer-like vector]   ← GREAT MATCH! 🎯
```

## Why Does This Work So Well?

**The vocabulary alignment problem:**
- Questions use interrogative language: "What is?", "How does?", "Why does?"
- Documents use declarative language: "X is...", "X works by...", "X occurs when..."
- These live in **different regions of embedding space**

HyDE **bridges that gap** by generating text that looks like the documents you're searching for.

## The HyDE Pipeline:

```
User Query
    ↓
LLM generates a hypothetical answer (doesn't need to be factually correct!)
    ↓
Embed the hypothetical answer
    ↓
Search knowledge base with that embedding
    ↓
Return real documents (the retrieved docs ARE factually correct)
```

⚠️ **Key insight:** The hypothetical answer doesn't need to be correct —  
it just needs to **sound like** the kind of document you're looking for.

## What You'll Learn

1. **Why queries and documents mismatch** in vector space
2. **HyDE step by step** — generate, embed, search
3. **Visualizing the difference** — query vs. answer vectors
4. **HyDE without an LLM** — template-based approximation
5. **HyDE with an LLM** — the real thing
6. **HyDE + Multi-Query** — combine for maximum recall
7. **When HyDE wins (and when it doesn't)**

---
**Prerequisites:** Basic embeddings + cosine similarity.  
Recommended: `02_Multi_Query_Generation.ipynb`

In [16]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()  # reads ANTHROPIC_API_KEY from .env if present

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def llm(prompt, max_tokens=400):
    """Call Claude Haiku. Fast and cheap — perfect for RAG pipelines."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

print("✅ Anthropic client ready!")
print("   Model: claude-haiku-4-5-20251001")
print("   Set ANTHROPIC_API_KEY in .env or environment.")


✅ Anthropic client ready!
   Model: claude-haiku-4-5-20251001
   Set ANTHROPIC_API_KEY in .env or environment.


## Setup

In [17]:
# Install if needed
# !pip install sentence-transformers numpy scikit-learn

In [18]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Ready!")

✅ Ready!


## 1. Build a Knowledge Base

In [19]:
# Knowledge base — these are the documents we want to retrieve
# Written in declarative, factual style (like a textbook or docs page)

documents = [
    # Overfitting & regularization
    "Overfitting occurs when a model learns the training data too well, capturing noise and failing to generalize to new data.",
    "Dropout randomly deactivates neurons during training, forcing the network to learn redundant representations and reducing overfitting.",
    "L2 regularization (weight decay) adds a penalty proportional to the square of weight values, discouraging large weights that cause overfitting.",
    "Early stopping monitors validation loss and halts training when it stops improving, preventing the model from memorizing training data.",
    "Data augmentation artificially expands the training set using transformations, giving the model more diverse examples and reducing overfitting.",

    # Transformers & attention
    "Self-attention allows a model to weigh the importance of each word in a sequence relative to all other words, capturing long-range dependencies.",
    "The transformer architecture replaces recurrence with attention mechanisms, enabling parallel computation and better handling of long sequences.",
    "Positional encoding adds position information to token embeddings because transformers have no built-in sense of word order.",

    # RAG
    "RAG (Retrieval-Augmented Generation) retrieves relevant documents at inference time and passes them as context to the language model.",
    "Chunking strategies in RAG split documents into overlapping segments to preserve context across chunk boundaries during retrieval.",
    "Embedding models in RAG convert both queries and documents into vectors so that semantic similarity can be measured by cosine distance.",

    # Training techniques
    "Transfer learning initializes a model with weights pretrained on a large dataset, then fine-tunes on a smaller task-specific dataset.",
    "Learning rate warmup gradually increases the learning rate at the start of training to prevent large, destabilizing gradient updates.",
    "Gradient clipping caps the magnitude of gradients during backpropagation to prevent exploding gradients in deep networks.",
]

# Embed all documents once
doc_embeddings = model.encode(documents)

print(f"Knowledge base: {len(documents)} documents")
print(f"Embedding shape: {doc_embeddings.shape}")

Knowledge base: 14 documents
Embedding shape: (14, 384)


## 2. The Query-Document Mismatch Problem

Let's **visualize** why questions and their answers live in different parts of vector space.

In [20]:
# Compare how similar a QUESTION is to its answer vs another question

# A question
question = "What causes overfitting in neural networks?"

# The actual answer (from our knowledge base)
answer = "Overfitting occurs when a model learns the training data too well, capturing noise and failing to generalize to new data."

# A different question (same format, different topic)
different_question = "Why do transformers need positional encoding?"

# Another question on the same topic as the first
similar_question = "How does a model overfit during training?"

# Embed all of them
q_emb           = model.encode(question)
a_emb           = model.encode(answer)
diff_q_emb      = model.encode(different_question)
similar_q_emb   = model.encode(similar_question)

# Compute cosine similarities
sim_q_to_answer        = cosine_similarity([q_emb], [a_emb])[0][0]
sim_q_to_diff_q        = cosine_similarity([q_emb], [diff_q_emb])[0][0]
sim_q_to_similar_q     = cosine_similarity([q_emb], [similar_q_emb])[0][0]

print("Cosine Similarity — how close are things in vector space?\n")
print(f"Question  vs  Its Answer       : {sim_q_to_answer:.4f}")
print(f"Question  vs  Similar Question : {sim_q_to_similar_q:.4f}")
print(f"Question  vs  Different Topic  : {sim_q_to_diff_q:.4f}")

print()
print("💡 Notice: The question is often CLOSER to another question than to its own answer!")
print("   This is the query-document mismatch problem that HyDE solves.")

Cosine Similarity — how close are things in vector space?

Question  vs  Its Answer       : 0.7548
Question  vs  Similar Question : 0.5762
Question  vs  Different Topic  : 0.1035

💡 Notice: The question is often CLOSER to another question than to its own answer!
   This is the query-document mismatch problem that HyDE solves.


In [21]:
# Let's confirm this with a real search

def search(query_text, top_k=5):
    """Basic embedding search — returns top_k results"""
    q_emb = model.encode(query_text)
    scores = cosine_similarity([q_emb], doc_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [
        {'rank': rank+1, 'doc_index': int(idx),
         'score': float(scores[idx]), 'document': documents[idx]}
        for rank, idx in enumerate(top_indices)
    ]


# Search with the plain question
query = "What causes overfitting in neural networks?"
results = search(query, top_k=5)

print(f"Query: '{query}'\n")
print("Results (plain question search):")
print("=" * 70)
for r in results:
    print(f"Rank {r['rank']} | {r['score']:.4f} | {r['document'][:65]}...")

# Check if the most relevant doc (index 0) ranked #1
print(f"\n🎯 Did the best overfitting doc (index 0) rank #1? ",
      "Yes ✅" if results[0]['doc_index'] == 0 else f"No ❌ — it ranked #{next(r['rank'] for r in results if r['doc_index']==0)}")

Query: 'What causes overfitting in neural networks?'

Results (plain question search):
Rank 1 | 0.7548 | Overfitting occurs when a model learns the training data too well...
Rank 2 | 0.5303 | Dropout randomly deactivates neurons during training, forcing the...
Rank 3 | 0.4540 | L2 regularization (weight decay) adds a penalty proportional to t...
Rank 4 | 0.4508 | Data augmentation artificially expands the training set using tra...
Rank 5 | 0.4201 | Gradient clipping caps the magnitude of gradients during backprop...

🎯 Did the best overfitting doc (index 0) rank #1?  Yes ✅


## 3. HyDE Step by Step

Now let's walk through the HyDE approach one step at a time.

In [22]:
# ── STEP 1: Start with the user's question ──

user_query = "What causes overfitting in neural networks?"
print(f"Step 1 — User Query: '{user_query}'")

Step 1 — User Query: 'What causes overfitting in neural networks?'


In [23]:
# ── STEP 2: Generate a hypothetical answer ──
# In production you'd call an LLM here.
# For now we write it manually to understand the concept.

# Notice: this sounds like a document, not a question!
hypothetical_answer = """
Overfitting in neural networks is caused by several factors:
training on too little data, using a model that is too complex for the task,
training for too many epochs, and insufficient regularization.
When a model overfits, it memorizes the training examples including their noise,
which causes high accuracy on training data but poor generalization to unseen data.
""".strip()

print(f"Step 2 — Hypothetical Answer:\n")
print(hypothetical_answer)

Step 2 — Hypothetical Answer:

Overfitting in neural networks is caused by several factors:
training on too little data, using a model that is too complex for the task,
training for too many epochs, and insufficient regularization.
When a model overfits, it memorizes the training examples including their noise,
which causes high accuracy on training data but poor generalization to unseen data.


In [24]:
# ── STEP 3: Embed the hypothetical answer ──

hyde_embedding = model.encode(hypothetical_answer)

print(f"Step 3 — Embedded the hypothetical answer")
print(f"         Vector shape: {hyde_embedding.shape}")

# Compare: is the hypothetical answer closer to the real document than the question was?
real_doc_emb = doc_embeddings[0]  # "Overfitting occurs when a model learns..."

sim_question_to_doc = cosine_similarity([model.encode(user_query)], [real_doc_emb])[0][0]
sim_hyde_to_doc     = cosine_similarity([hyde_embedding], [real_doc_emb])[0][0]

print(f"\n         Similarity: Original question → real doc  : {sim_question_to_doc:.4f}")
print(f"         Similarity: HyDE answer      → real doc  : {sim_hyde_to_doc:.4f}")
print(f"\n         ✅ HyDE is {sim_hyde_to_doc - sim_question_to_doc:.4f} closer to the real document!")

Step 3 — Embedded the hypothetical answer
         Vector shape: (384,)

         Similarity: Original question → real doc  : 0.7548
         Similarity: HyDE answer      → real doc  : 0.8327

         ✅ HyDE is 0.0779 closer to the real document!


In [25]:
# ── STEP 4: Search with the hypothetical answer embedding ──

scores = cosine_similarity([hyde_embedding], doc_embeddings)[0]
top_indices = np.argsort(scores)[::-1][:5]

print(f"Step 4 — Search results using HyDE embedding:\n")
print("=" * 70)
for rank, idx in enumerate(top_indices, 1):
    print(f"Rank {rank} | {scores[idx]:.4f} | {documents[idx][:65]}...")

Step 4 — Search results using HyDE embedding:

Rank 1 | 0.8327 | Overfitting occurs when a model learns the training data too well...
Rank 2 | 0.5225 | Dropout randomly deactivates neurons during training, forcing the...
Rank 3 | 0.5096 | Data augmentation artificially expands the training set using tra...
Rank 4 | 0.4411 | L2 regularization (weight decay) adds a penalty proportional to t...
Rank 5 | 0.3613 | Gradient clipping caps the magnitude of gradients during backprop...


In [26]:
# ── Compare normal search vs HyDE side-by-side ──

normal_results = search(user_query, top_k=5)

print("COMPARISON: Normal Search  vs  HyDE\n")
print(f"Query: '{user_query}'\n")

print(f"{'Normal Search':<45} {'HyDE Search'}")
print("-" * 90)

for rank in range(5):
    n = normal_results[rank]
    h_idx = top_indices[rank]
    h_score = scores[h_idx]
    
    n_text = n['document'][:40]
    h_text = documents[h_idx][:40]
    
    print(f"{n['score']:.3f} {n_text:<42} {h_score:.3f} {h_text}")

COMPARISON: Normal Search  vs  HyDE

Query: 'What causes overfitting in neural networks?'

Normal Search                                 HyDE Search
------------------------------------------------------------------------------------------
0.755 Overfitting occurs when a model learns t   0.833 Overfitting occurs when a model learns t
0.530 Dropout randomly deactivates neurons dur   0.523 Dropout randomly deactivates neurons dur
0.454 L2 regularization (weight decay) adds a    0.510 Data augmentation artificially expands t
0.451 Data augmentation artificially expands t   0.441 L2 regularization (weight decay) adds a 
0.420 Gradient clipping caps the magnitude of    0.361 Gradient clipping caps the magnitude of 


## 4. HyDE Without an LLM — Template Approximation

No LLM? No problem. We can use **simple templates** to convert a question into answer-style text.

It's not as good as a real LLM, but it's free and fast.

In [27]:
# Template-based HyDE approximation
# We transform the question into declarative phrasing

def template_hyde(question):
    """
    Approximate HyDE by transforming question phrasing into answer phrasing.
    Simple string replacements convert interrogative → declarative.
    """
    # Strip question mark
    q = question.rstrip('?').strip()
    
    # Common transformations (question → statement)
    transformations = [
        ("What causes ",   ""),               # "What causes X" → "X"
        ("What is ",       ""),               # "What is X" → "X"
        ("What are ",      ""),               # "What are X" → "X"
        ("How does ",      ""),               # "How does X" → "X"
        ("How do ",        ""),               # "How do X" → "X"
        ("Why does ",      "The reason "),    # "Why does X" → "The reason X"
        ("Why do ",        "The reason "),
        ("When should ",   ""),
        ("How can ",       ""),
    ]
    
    for pattern, replacement in transformations:
        if q.startswith(pattern):
            q = replacement + q[len(pattern):]
            break
    
    # Build a declarative sentence
    return f"{q}. This is an important concept in machine learning and AI."


# Test with several questions
test_questions = [
    "What causes overfitting in neural networks?",
    "How does self-attention work in transformers?",
    "Why does gradient clipping help during training?",
    "What is transfer learning?",
]

print("Template HyDE transformations:\n")
for q in test_questions:
    transformed = template_hyde(q)
    print(f"  Q: {q}")
    print(f"  A: {transformed}")
    print()

Template HyDE transformations:

  Q: What causes overfitting in neural networks?
  A: overfitting in neural networks. This is an important concept in machine learning and AI.

  Q: How does self-attention work in transformers?
  A: self-attention work in transformers. This is an important concept in machine learning and AI.

  Q: Why does gradient clipping help during training?
  A: The reason gradient clipping help during training. This is an important concept in machine learning and AI.

  Q: What is transfer learning?
  A: transfer learning. This is an important concept in machine learning and AI.



In [28]:
# Now search using template-based HyDE

def template_hyde_search(question, top_k=5):
    """Search using template-based HyDE approximation"""
    
    # Step 1: Transform question to answer-like text
    hypothetical = template_hyde(question)
    
    # Step 2: Embed the hypothetical answer
    hyde_emb = model.encode(hypothetical)
    
    # Step 3: Search
    scores = cosine_similarity([hyde_emb], doc_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    return [
        {'rank': rank+1, 'doc_index': int(idx),
         'score': float(scores[idx]), 'document': documents[idx],
         'hypothetical': hypothetical}
        for rank, idx in enumerate(top_indices)
    ]


# Test it
question = "What causes overfitting in neural networks?"
results = template_hyde_search(question)

print(f"Question: '{question}'")
print(f"Hypothetical: '{results[0]['hypothetical']}'\n")
print("Template HyDE Results:")
print("=" * 70)
for r in results:
    print(f"Rank {r['rank']} | {r['score']:.4f} | {r['document'][:65]}...")

Question: 'What causes overfitting in neural networks?'
Hypothetical: 'overfitting in neural networks. This is an important concept in machine learning and AI.'

Template HyDE Results:
Rank 1 | 0.7199 | Overfitting occurs when a model learns the training data too well...
Rank 2 | 0.5381 | Data augmentation artificially expands the training set using tra...
Rank 3 | 0.4940 | Dropout randomly deactivates neurons during training, forcing the...
Rank 4 | 0.3819 | Gradient clipping caps the magnitude of gradients during backprop...
Rank 5 | 0.3669 | L2 regularization (weight decay) adds a penalty proportional to t...


## 5. HyDE With an LLM — The Real Thing

This is where HyDE really shines. The LLM writes a rich, fluent hypothetical answer
that deeply matches the vocabulary and style of your documents.

In [29]:
# The HyDE prompt you'd send to an LLM
# We print it here so you understand exactly what to ask for

def build_hyde_prompt(question):
    return f"""Write a short, factual paragraph (2-4 sentences) that answers the following question.
Write it in a neutral, encyclopedic style — as if writing a textbook or documentation.
Do not say 'I' or address the user. Just state the facts directly.

Question: {question}

Answer:"""


# Print the prompt so you can see exactly what goes to the LLM
question = "What causes overfitting in neural networks?"
print("Prompt sent to LLM:")
print("-" * 50)
print(build_hyde_prompt(question))

Prompt sent to LLM:
--------------------------------------------------
Write a short, factual paragraph (2-4 sentences) that answers the following question.
Write it in a neutral, encyclopedic style — as if writing a textbook or documentation.
Do not say 'I' or address the user. Just state the facts directly.

Question: What causes overfitting in neural networks?

Answer:


In [30]:
# Generate a hypothetical answer using Claude
# This is the core of HyDE — the LLM writes an answer-like text for better search

def generate_hypothetical_answer(question):
    prompt = build_hyde_prompt(question)
    return llm(prompt, max_tokens=150)


# Test it
question = "What causes overfitting in neural networks?"
hypothetical = generate_hypothetical_answer(question)

print(f"Question: '{question}'\n")
print("LLM-generated hypothetical answer:")
print("-" * 50)
print(hypothetical)


Question: 'What causes overfitting in neural networks?'

LLM-generated hypothetical answer:
--------------------------------------------------
# Overfitting in Neural Networks

Overfitting occurs in neural networks when a model learns the training data too well, including its noise and idiosyncrasies, rather than learning generalizable patterns. This typically happens when a model has excessive capacity relative to the amount of training data, allowing it to memorize specific examples instead of capturing underlying relationships. Common causes include insufficient training data, models with too many parameters, inadequate regularization, and training for too many epochs without early stopping. As a result, the model performs well on training data but poorly on unseen test data.


In [31]:
# Full LLM-based HyDE search

def llm_hyde_search(question, top_k=5):
    """HyDE search using a Claude-generated hypothetical answer"""

    # Step 1: Build the prompt
    prompt = build_hyde_prompt(question)

    # Step 2: Generate hypothetical answer with Claude
    hypothetical = llm(prompt, max_tokens=150)

    # Step 3: Embed the hypothetical answer
    hyde_emb = model.encode(hypothetical)

    # Step 4: Search using that embedding
    scores = cosine_similarity([hyde_emb], doc_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [                        
        {'rank': rank+1, 'doc_index': int(idx),
         'score': float(scores[idx]), 'document': documents[idx],
         'hypothetical': hypothetical}
        for rank, idx in enumerate(top_indices)
    ]


# Run it
question = "What causes overfitting in neural networks?"
hyde_results = llm_hyde_search(question)

print(f"Query: '{question}'")
print(f"\nHypothetical answer used for search:")
print(f"  '{hyde_results[0]['hypothetical'][:100]}...'\n")

print("LLM HyDE Search Results:")
print("=" * 70)
for r in hyde_results:
    print(f"Rank {r['rank']} | {r['score']:.4f} | {r['document'][:65]}...")


Query: 'What causes overfitting in neural networks?'

Hypothetical answer used for search:
  '# Overfitting in Neural Networks

Overfitting occurs in neural networks when a model learns the trai...'

LLM HyDE Search Results:
Rank 1 | 0.7822 | Overfitting occurs when a model learns the training data too well...
Rank 2 | 0.5439 | Dropout randomly deactivates neurons during training, forcing the...
Rank 3 | 0.5068 | L2 regularization (weight decay) adds a penalty proportional to t...
Rank 4 | 0.4487 | Data augmentation artificially expands the training set using tra...
Rank 5 | 0.3744 | Early stopping monitors validation loss and halts training when i...


## 6. Full Three-Way Comparison

Let's compare all three approaches across multiple questions.

In [32]:
# Compare: Normal vs Template HyDE vs LLM HyDE

test_questions = [
    "What causes overfitting in neural networks?",
    "How does self-attention work in transformers?",
    "How does RAG reduce hallucination?",
    "What is transfer learning?",
]

for question in test_questions:
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print(f"{'='*70}")
    
    # Method 1: Normal search
    normal = search(question, top_k=1)
    
    # Method 2: Template HyDE
    template = template_hyde_search(question, top_k=1)
    
    # Method 3: LLM HyDE
    llm_hyde = llm_hyde_search(question, top_k=1)
    
    print(f"\n  Normal Search  [{normal[0]['score']:.4f}]:   {normal[0]['document'][:60]}...")
    print(f"  Template HyDE  [{template[0]['score']:.4f}]:   {template[0]['document'][:60]}...")
    print(f"  LLM HyDE       [{llm_hyde[0]['score']:.4f}]:   {llm_hyde[0]['document'][:60]}...")

print("\n💡 LLM HyDE consistently surfaces the most relevant document with highest similarity!")


QUESTION: What causes overfitting in neural networks?

  Normal Search  [0.7548]:   Overfitting occurs when a model learns the training data too...
  Template HyDE  [0.7199]:   Overfitting occurs when a model learns the training data too...
  LLM HyDE       [0.8026]:   Overfitting occurs when a model learns the training data too...

QUESTION: How does self-attention work in transformers?

  Normal Search  [0.5587]:   Self-attention allows a model to weigh the importance of eac...
  Template HyDE  [0.5652]:   Self-attention allows a model to weigh the importance of eac...
  LLM HyDE       [0.6845]:   Self-attention allows a model to weigh the importance of eac...

QUESTION: How does RAG reduce hallucination?

  Normal Search  [0.2655]:   Chunking strategies in RAG split documents into overlapping ...
  Template HyDE  [0.3235]:   Dropout randomly deactivates neurons during training, forcin...
  LLM HyDE       [0.7543]:   RAG (Retrieval-Augmented Generation) retrieves relevant docu...

Q

## 7. HyDE + Multi-Query — The Power Combo

You can combine HyDE with multi-query generation for maximum coverage:
- **Multi-query** catches documents with different vocabulary
- **HyDE** bridges the question-to-document gap

Together they're significantly more powerful than either alone.

In [33]:
# HyDE + Multi-Query combined pipeline

def reciprocal_rank_fusion(query_results_list, k=60):
    """Merge multiple result lists using RRF (from previous notebook)"""
    rrf_scores = {}
    for results in query_results_list:
        for r in results:
            idx = r['doc_index']
            rank = r['rank']
            rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)


def hyde_multi_query_search(question, top_k=5):
    """
    HyDE + Multi-Query combined:
    1. Normal search (original question)
    2. Template HyDE (question → declarative statement)
    3. LLM HyDE (question → rich hypothetical answer)
    4. Merge all with RRF
    """
    all_result_sets = []
    
    # Search 1: plain question
    all_result_sets.append(search(question, top_k=10))
    
    # Search 2: template HyDE
    all_result_sets.append(template_hyde_search(question, top_k=10))
    
    # Search 3: LLM HyDE
    all_result_sets.append(llm_hyde_search(question, top_k=10))
    
    # Merge with RRF
    rrf_ranked = reciprocal_rank_fusion(all_result_sets)
    
    return [
        {'rank': i+1, 'doc_index': idx, 'rrf_score': score, 'document': documents[idx]}
        for i, (idx, score) in enumerate(rrf_ranked[:top_k])
    ]


# Test the combo
question = "What causes overfitting in neural networks?"
combo_results = hyde_multi_query_search(question, top_k=5)

print(f"HyDE + Multi-Query Results for:")
print(f"'{question}'\n")
print("=" * 70)
for r in combo_results:
    print(f"Rank {r['rank']} | RRF: {r['rrf_score']:.4f} | {r['document'][:65]}...")

HyDE + Multi-Query Results for:
'What causes overfitting in neural networks?'

Rank 1 | RRF: 0.0492 | Overfitting occurs when a model learns the training data too well...
Rank 2 | RRF: 0.0481 | Dropout randomly deactivates neurons during training, forcing the...
Rank 3 | RRF: 0.0476 | Data augmentation artificially expands the training set using tra...
Rank 4 | RRF: 0.0469 | L2 regularization (weight decay) adds a penalty proportional to t...
Rank 5 | RRF: 0.0462 | Gradient clipping caps the magnitude of gradients during backprop...


In [34]:
# Final score: how many unique documents does each method surface in top-5?

question = "What causes overfitting in neural networks?"

normal_top5  = {r['doc_index'] for r in search(question, top_k=5)}
template_top5 = {r['doc_index'] for r in template_hyde_search(question, top_k=5)}
llm_top5     = {r['doc_index'] for r in llm_hyde_search(question, top_k=5)}
combo_top5   = {r['doc_index'] for r in hyde_multi_query_search(question, top_k=5)}

# Ground truth: overfitting docs are indices 0–4
ground_truth = {0, 1, 2, 3, 4}

def recall_at_5(retrieved, relevant):
    return len(retrieved & relevant) / len(relevant)

print("Recall@5 (how many of the 5 relevant docs did we find?)\n")
print(f"  Normal Search         : {recall_at_5(normal_top5,   ground_truth):.0%}")
print(f"  Template HyDE         : {recall_at_5(template_top5, ground_truth):.0%}")
print(f"  LLM HyDE              : {recall_at_5(llm_top5,      ground_truth):.0%}")
print(f"  HyDE + Multi-Query    : {recall_at_5(combo_top5,    ground_truth):.0%}")
print("\n🏆 Combining methods gives the best recall!")

Recall@5 (how many of the 5 relevant docs did we find?)

  Normal Search         : 80%
  Template HyDE         : 80%
  LLM HyDE              : 100%
  HyDE + Multi-Query    : 80%

🏆 Combining methods gives the best recall!


## 8. When HyDE Wins (and When It Doesn't)

In [35]:
guide = """
┌──────────────────────────────────────────────────────────────────┐
│                    HyDE Decision Guide                           │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  ✅ HyDE WINS when:                                              │
│  • Queries are questions ("What is?", "How does?")               │
│  • Documents are factual / encyclopedic (textbooks, docs, FAQs)  │
│  • Users phrase things differently than documents                │
│  • You need high precision on Q&A tasks                          │
│                                                                  │
│  ❌ HyDE STRUGGLES when:                                         │
│  • Query is already a keyword / noun phrase ("RAG", "dropout")   │
│  • Documents are very short (tweet-length, titles)               │
│  • The LLM generates a confidently wrong hypothetical            │
│  • Latency budget is very tight (adds 1 LLM call)                │
│                                                                  │
│  ⚡ LATENCY:                                                     │
│  • Template HyDE : +0ms (just string ops + 1 extra encode)       │
│  • LLM HyDE      : +200-600ms for the LLM call                   │
│                                                                  │
│  💰 COST:                                                        │
│  • Template HyDE : free                                          │
│  • LLM HyDE      : ~$0.0001 per query with a small model         │
│                                                                  │
│  📈 TYPICAL IMPROVEMENT: +5% to +25% precision on Q&A tasks      │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
"""

print(guide)


┌──────────────────────────────────────────────────────────────────┐
│                    HyDE Decision Guide                           │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  ✅ HyDE WINS when:                                              │
│  • Queries are questions ("What is?", "How does?")               │
│  • Documents are factual / encyclopedic (textbooks, docs, FAQs)  │
│  • Users phrase things differently than documents                │
│  • You need high precision on Q&A tasks                          │
│                                                                  │
│  ❌ HyDE STRUGGLES when:                                         │
│  • Query is already a keyword / noun phrase ("RAG", "dropout")   │
│  • Documents are very short (tweet-length, titles)               │
│  • The LLM generates a confidently wrong hypothetical            │
│  • Latency budget is very tight (

## Key Takeaways 🎯

### What You Learned:

| Concept | What It Does |
|---|---|
| **Query-document mismatch** | Questions and answers live in different vector regions |
| **HyDE** | Generates a hypothetical answer to bridge that gap |
| **Template HyDE** | Free approximation — transforms question phrasing into answer phrasing |
| **LLM HyDE** | Gold standard — LLM writes a rich, vocabulary-aligned answer |
| **HyDE + Multi-Query** | Highest recall — cast the widest, most targeted net |

### The HyDE Recipe:

```python
# 1. User asks a question
question = "What causes overfitting?"

# 2. LLM generates a hypothetical answer
hypothetical = llm.generate(hyde_prompt(question))

# 3. Embed the answer (not the question!)
embedding = embed(hypothetical)

# 4. Search with the answer embedding
results = vector_search(embedding, knowledge_base)

# 5. Return real documents (factually grounded)
return results
```

### Key Insight:

> **The hypothetical answer doesn't need to be factually correct.**  
> It just needs to sound like the documents you're searching for.  
> The retrieved documents are the source of truth — not the LLM's guess.

---

## What's Next? 🔜

**`04_Query_Decomposition.ipynb`** — For complex, multi-part questions, break them into sub-questions, answer each separately, then combine:

```
"Compare dropout and L2 regularization for preventing overfitting"
             ↓  decompose
  Sub-query 1: "What is dropout and how does it work?"
  Sub-query 2: "What is L2 regularization and how does it work?"
  Sub-query 3: "Comparison of regularization methods"
             ↓  retrieve each separately
             ↓  combine contexts
  → Much better answer than one-shot retrieval!
```